[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pydantic-certified/notebooks/day-01-basemodel-basics.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · BaseModel Basics — Pydantic v2 from the Ground Up
**certified-journeys / pydantic-certified** · Day 1 · Foundation

> **Goal for today:** By the end of this notebook you can define a Pydantic v2 `BaseModel`, validate data with it, and inspect `ValidationError` details when invalid data is passed.

In [ ]:
%pip install -q 'pydantic>=2.0' pydantic-settings

## Step 1 · Verify Pydantic v2 is installed

Pydantic v2 ships with **pydantic-core**, a Rust extension that makes validation 5–50× faster than v1.
Before writing any models, confirm the installed version is `2.x`.

| Version | Core | Speed vs v1 |
|---------|------|-------------|
| v1 | Pure Python | baseline |
| v2 | Rust (pydantic-core) | 5–50× faster |

In [ ]:
import pydantic

# Print the installed version — should be 2.x
print("Pydantic version:", pydantic.__version__)

# pydantic-core is the Rust extension bundled with v2
import pydantic_core
print("pydantic-core version:", pydantic_core.__version__)

**What just happened?**

- We imported `pydantic` and confirmed we have v2 installed.
- `pydantic_core` is the Rust-backed validation engine — it ships automatically with Pydantic v2.
- **If you see `1.x`**, run `%pip install -q 'pydantic>=2.0'` again and restart the runtime.
- The version string format changed from `1.10.x` → `2.x.y`, so any `2.*` is correct.

## Step 2 · Define your first BaseModel

A `BaseModel` subclass turns a plain Python class into a validated data container.
You annotate fields with standard Python type hints — no extra decorators needed.

```
class MyModel(BaseModel):
    field_name: type
    optional_field: type = default_value
```

Pydantic reads the annotations at **class creation time** and builds a schema. Validation
happens when you call `MyModel(...)` or `MyModel.model_validate({...})`.

In [ ]:
from pydantic import BaseModel

# Define a User model with three typed fields
class User(BaseModel):
    name: str           # required — no default
    age: int            # required — must be an integer
    email: str          # required — plain string for now (Day 3 adds email validation)

# Create a valid instance using keyword arguments
user = User(name="Alice", age=30, email="alice@example.com")
print(user)

# Access fields like regular Python attributes
print("Name:", user.name)
print("Age:", user.age)

# .model_dump() returns a plain dict — useful for serialisation
print("Dict:", user.model_dump())

**What just happened?**

- `User(name=..., age=..., email=...)` triggers Pydantic's validation pipeline.
- **All three fields are required** — omitting any one raises a `ValidationError`.
- `user.model_dump()` is the v2 replacement for v1's `.dict()` — keep this change in mind when migrating.
- Fields are exposed as normal Python attributes; there is no magic proxy layer.

## Step 3 · Pydantic coerces compatible types automatically (lax mode)

By default, Pydantic runs in **lax mode**: it tries sensible coercions before rejecting a value.
For example, a string `"30"` is coerced to `int` 30 for an `age: int` field.

| Input | Field type | Result (lax) |
|-------|-----------|---------------|
| `"30"` | `int` | `30` (coerced) |
| `30.9` | `int` | `30` (truncated) |
| `"alice"` | `int` | `ValidationError` |
| `42` | `str` | `"42"` (coerced) |

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int
    email: str

# String "30" is coerced to int 30 in lax (default) mode
u1 = User(name="Bob", age="30", email="bob@example.com")
print("age type:", type(u1.age), "value:", u1.age)  # <class 'int'> 30

# Integer 42 is coerced to string "42" for a str field
u2 = User(name=42, age=25, email="x@y.com")
print("name type:", type(u2.name), "value:", u2.name)  # <class 'str'> '42'

# model_validate() accepts a dict — same validation pipeline
u3 = User.model_validate({"name": "Carol", "age": "28", "email": "c@ex.com"})
print("u3 via model_validate:", u3)

**What just happened?**

- Lax mode makes Pydantic forgiving about *compatible* types — great for API data that arrives as strings.
- `model_validate(dict)` is the v2 replacement for v1's `Model.parse_obj(dict)`.
- **Coercion is one-way**: `"30"` → `int` works; `"hello"` → `int` still raises `ValidationError`.
- Strict mode (Day 2) disables all coercion for tighter contracts.

## Step 4 · Trigger and catch a ValidationError

When Pydantic cannot coerce a value to the required type, it raises `pydantic.ValidationError`.
**Crucially, Pydantic collects *all* field errors before raising** — you see every problem at once,
not just the first one.

```python
from pydantic import ValidationError
try:
    Model(bad_field=bad_value)
except ValidationError as e:
    print(e.errors())   # list of dicts, one per error
```

In [ ]:
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    name: str
    age: int
    email: str

# Pass multiple invalid fields so we can see all errors at once
try:
    bad_user = User(
        name="Dave",
        age="not-a-number",   # cannot coerce to int
        email="dave@ok.com",
    )
except ValidationError as e:
    print("Number of errors:", e.error_count())
    print()
    # e.errors() returns a list of error detail dicts
    for err in e.errors():
        print("Field  :", err["loc"])    # tuple of field path, e.g. ('age',)
        print("Message:", err["msg"])    # human-readable message
        print("Type   :", err["type"])   # machine-readable error code
        print()

**What just happened?**

- `ValidationError` is the single exception type for all Pydantic validation failures.
- `.errors()` returns a `list[dict]` — each dict has `loc` (field path), `msg`, `type`, and `input`.
- `e.error_count()` is a convenient shorthand for `len(e.errors())`.
- **`type` is a stable string code** (e.g. `"int_parsing"`) — use it in tests instead of parsing the human message.

## Step 5 · Missing required fields also raise ValidationError

Fields without a default value are **required**. Omitting them raises `ValidationError` with
error type `"missing"`. This is different from passing the wrong type — the `loc` is the field
name and the message is `"Field required"`.

In [ ]:
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    name: str
    age: int
    email: str

# Omit 'age' and 'email' — both should appear in the error list
try:
    incomplete = User(name="Eve")
except ValidationError as e:
    print("Validation errors when fields are missing:")
    print(e)  # pretty-printed by default
    print()
    # Each missing field produces a separate error dict
    for err in e.errors():
        print(f"  {err['loc'][0]!r}: {err['msg']} (type={err['type']!r})")

print()
# A model with an optional field (has a default) does NOT raise for that field
class UserWithOptional(BaseModel):
    name: str
    age: int
    email: str = "unknown@example.com"   # default → optional

u = UserWithOptional(name="Frank", age=22)
print("email defaulted to:", u.email)

**What just happened?**

- Pydantic reports *all* missing fields in one `ValidationError`, not just the first.
- The error type for missing fields is `"missing"` — consistent and easy to test.
- **Adding a default value** (`email: str = "..."`) makes the field optional — no error if omitted.
- `print(e)` gives a nicely formatted multi-line summary; `.errors()` gives structured data for programmatic handling.

## Step 6 · Inspect model schema and model_json_schema

Pydantic v2 can export a JSON Schema for your model. This is useful for:
- Generating OpenAPI documentation automatically
- Sharing contracts with other teams
- Validating data with external tools

`Model.model_json_schema()` returns a Python dict following JSON Schema Draft 2020-12.

In [ ]:
import json
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int
    email: str

# model_json_schema() — v2 replacement for v1's .schema()
schema = User.model_json_schema()
print(json.dumps(schema, indent=2))

print()
# model_fields tells you about each field: type, required, default
for field_name, field_info in User.model_fields.items():
    print(f"  {field_name}: required={field_info.is_required()}, annotation={field_info.annotation}")

**What just happened?**

- `model_json_schema()` replaces v1's `.schema()` — both produce JSON Schema, but v2 targets Draft 2020-12.
- `model_fields` is a `dict[str, FieldInfo]` — inspect it in tests to assert field constraints.
- **The schema is derived from type annotations at class creation time**, not from instances.
- FastAPI uses `model_json_schema()` internally to generate its OpenAPI spec.

In [ ]:
# Challenge: Build a Product model and demonstrate validation
#
# 1. Define a Product model with fields:
#    - product_id: int
#    - name: str
#    - price: float
#    - in_stock: bool (default True)
#
# 2. Create a valid Product instance and print model_dump()
#
# 3. Try to create a Product with price="free" and catch the
#    ValidationError — print the error type for the price field
#
# 4. Print the JSON schema for Product
#
# Your solution here


---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `BaseModel` | Subclass it; use Python type hints for fields |
| Required vs optional | No default → required; any default → optional |
| Lax mode (default) | Compatible types are coerced (e.g. `"30"` → `int`) |
| `ValidationError` | Raised with *all* errors collected; inspect with `.errors()` |
| `model_dump()` | v2 replacement for v1's `.dict()` |
| `model_validate()` | v2 replacement for v1's `.parse_obj()` |
| `model_json_schema()` | v2 replacement for v1's `.schema()` |
| `pydantic-core` | Rust engine bundled with v2 — automatic speed boost |

> **Tip:** Pydantic v2 ships with a Rust core (pydantic-core) that makes validation 5–50× faster than v1. You get the speed boost automatically — no code changes needed, just upgrade.

---
## What's next
**Day 2** → Fields, Types, and Defaults — use `Field()`, constrained types (`PositiveInt`, `constr`), aliases, and strict mode to build precise, reusable schemas.

Mark Day 1 complete in your [tracker](../index.html).